# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an end-to-end guide for loading, exploring, and processing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema with multiple record sets and fields. Entities in the dataset (record sets, fields, columns, etc.) are **always referenced by their `@id`** according to the Croissant specification.

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
List all available record sets, their `@id`, and fields in the dataset. This helps build the mapping for later data loading.

We'll use the Croissant API to print all record set `@id`s and their respective fields, also by their `@id`.

In [ ]:

record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name} | @id: {rs.id if hasattr(rs, 'id') else getattr(rs, '@id', '?')}")
    # List fields/columns for each record set
    if hasattr(rs, "fields"):
        print("  Fields:")
        for fld in rs.fields:
            col_id = getattr(fld, 'id', getattr(fld, '@id', '?'))
            col_name = getattr(fld, 'name', getattr(fld, 'label', '?'))
            print(f"    - {col_name} | @id: {col_id}")
    print()

## 3. Data Extraction

Select the main data record set by its `@id` (see above) and load it as a DataFrame.

It's common for mass spectrometry datasets to have one primary tabular record set (such as proteins or peptides). Below, we extract all main record sets, but focus on the principal table record set for further exploration.

In [ ]:
# STEP 1: List all record set @ids again for convenience
rs_ids = [getattr(rs, 'id', getattr(rs, '@id', None)) for rs in dataset.record_sets()]
print(f"Record set @ids: {rs_ids}\n")

# STEP 2: Select main record set (replace this with your main @id, review previous cell output)
# In this FAIR^2 schema, the main table often uses the @id ending with 'protein_table', but print above to confirm it!
main_record_set_id = rs_ids[0] if len(rs_ids) == 1 else rs_ids[0]  # Assuming single main table, modify if needed
print(f"Using main record set: {main_record_set_id}\n")

# STEP 3: Load record set(s) as DataFrames
dataframes = {}
for rs_id in rs_ids:
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    dataframes[rs_id] = df
    print(f"Loaded record set {rs_id}: {df.shape[0]} rows, {df.shape[1]} columns.")
    print(f"Columns: {df.columns.tolist()}\n")

# For further EDA focus on main_record_set_id
main_df = dataframes[main_record_set_id]
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field—commonly something like `abundance`, `coverage`, or `peptide_count`. Please see your columns for the exact name (found above: e.g., `cr:Abundance` or similar).

We'll filter for proteins with high abundance, normalize abundances, and optionally group by a key field (e.g., experiment or sample).

In [ ]:
# Pick a numeric field and a group field from column names printed above
# MODIFY these to match your actual field @id (e.g., 'cr:Abundance', 'cr:Coverage', etc.)
numeric_field = None
possible_numeric = [col for col in main_df.columns if any(kw in col.lower() for kw in ["abundance", "coverage", "peptide", "count", "mw", "mass", "score"])]
if possible_numeric:
    numeric_field = possible_numeric[0]  # We'll use the first matching numeric field
print(f"Chosen numeric field: {numeric_field}")

if numeric_field:
    # Drop NA and ensure numeric
    col = main_df[numeric_field].astype(float)
    threshold = col.mean() if col.mean() > 0 else 10
    filtered_df = main_df[col > threshold].copy()
    print(f"Records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} proteins\n")

    filtered_df[numeric_field + "_normalized"] = (filtered_df[numeric_field].astype(float) - col.mean()) / col.std()
    print(f"Sample normalized values:")
    print(filtered_df[[numeric_field, numeric_field + "_normalized"]].head(), "\n")

    # If grouping field is available, group (e.g., by 'sample' or 'condition')
    possible_group = [col for col in main_df.columns if any(kw in col.lower() for kw in ["sample", "experiment", "condition", "replicate"])]
    group_field = possible_group[0] if possible_group else None
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped mean {numeric_field}_normalized by {group_field}:")
        print(grouped_df[[numeric_field + "_normalized"]].head(), "\n")
else:
    print("No numeric field found for analysis. Adjust field names above as needed.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field (e.g., abundance) and, if possible, a boxplot grouped by sample/condition. Visualization depends on field availability.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field].astype(float), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if available
    if 'group_field' in locals() and group_field and group_field in main_df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=main_df[group_field], y=main_df[numeric_field].astype(float))
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Skip visualization: No numeric field selected.")

## 6. Conclusion

* We have loaded and explored the dataset schema and data records using the Croissant specification and `mlcroissant`.
* The dataset contains key measurement fields such as protein abundance, peptide count, coverage, etc. (see column names above).
* We performed basic filtering, normalization, and grouping by attributes (sample or experiment, if available).
* Visualization highlights the distribution and sample variability in protein quantification.

Further analysis could include comparing additional samples, annotating proteins with metadata, or connecting to UniProt for biological enrichment. Always refer to entities and fields with their Croissant `@id` for reproducibility.